# Lesson 3.4 - The ledger

**Module 3 - Ingest.** The notebook for the second How beat: the code of `services/ingest/idempotency.py` and the two entry points in `main.py` that call it.

**What this notebook proves**

1. The three functions the lesson reads most - `claim()`, `swap_versions()`, `reactivate()` - printed from the kit's own source with line numbers and annotated line by line.
2. The claim race: two "workers" on a plain dict without a transaction index one version twice; with a lock, once.
3. `reactivate()`'s verdict for four (rows, chunks, age) cases, using the same two rules copied from the function.
4. On a deployed lane: the ledger rows of `acme`, one claim, `ledger/acme`, the corpus fingerprint recomputed with the kit's own `corpus_fingerprint()` and compared with the stored one, and current versus retired rows per source.

**What it costs.** Cells 1 to 9 are Rs 0: they read source and run pure Python. The last four cells need a deployed lane from Module 2 and cost a few Firestore reads (well under a rupee). Nothing here embeds, parses or uploads anything.

In [ ]:
%pip install -q pydantic google-cloud-firestore

## The kit

The public learner repo mirrors the kit. Clone it once; the kit folder inside it is `deploy/` today (`code/` after the rename), and the cell finds either. `idempotency.py` imports only `hashlib`, `os`, `datetime` and the Firestore client, so it imports without a project.

In [ ]:
import os, subprocess, sys
KIT_BRANCH = "main"
if not os.path.isdir("agentic-ai-weekend-gcp-learners"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", KIT_BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners"], check=True)
KIT = next(p for p in ("agentic-ai-weekend-gcp-learners/deploy", "agentic-ai-weekend-gcp-learners/code")
           if os.path.isdir(os.path.join(p, "services", "ingest")))
for p in (KIT, os.path.join(KIT, "services", "ingest")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("kit:", KIT)
import idempotency
if not hasattr(idempotency, "RETENTION_DAYS"):
    print("WARNING: this kit copy predates the ledger of 12 September 2026 (no RETENTION_DAYS, no retake, no withdrawn):")
    print("         the learner repo has not been rebuilt from the curriculum yet; the annotations below will not match.")

## Auth

Only the last four cells touch a project. Application Default Credentials, never an API key; on Colab the cell opens the sign-in, elsewhere run `gcloud auth application-default login` first. Skip it until you reach the live cells if you have no lane.

In [ ]:
PROJECT_ID = "documind-ai-YOUR-ID"
try:
    from google.colab import auth; auth.authenticate_user()
except ImportError:
    pass   # not on Colab: use `gcloud auth application-default login`

## 1. The source, annotated - `claim()`

`inspect.getsource` reads the function off the kit file you just cloned, so what prints is what runs on the lane. The helper numbers every line from the file and adds a note where the lesson's step 2 points.

In [ ]:
import inspect
import idempotency

def annotate(fn, notes):
    lines, start = inspect.getsourcelines(fn)
    for n, line in enumerate(lines, start):
        note = next((v for k, v in notes.items() if k in line), "")
        print(f"{n:4}  {line.rstrip()}" + (f"      # <- {note}" if note else ""))

annotate(idempotency.claim, {
    "retakes = ": "the retake rule: failed always, superseded only when the caller asks, queued never",
    "@firestore.transactional": "the whole point: the read and the write below are one unit",
    "not in retakes": "somebody has it, or had it and finished: refuse",
    "tx.set(ref": "the claim row: tenant_id as a field, status processing",
    "ingest_duplicate": "logged on EVERY refusal, before push() decides undo / queued / duplicate",
})

## 2. The source, annotated - `swap_versions()`

Two loops in one batch: the new version's rows go current first, then every other current row of the source is retired. The order is the design (lesson step 5).

In [ ]:
annotate(idempotency.swap_versions, {
    "rows = list(query.stream())": "every row of this source, any version, any flag",
    "# the new version first": "loop 1: a reader never finds NO version",
    '"staged": firestore.DELETE_FIELD': "the stage marks and any old retirement fields go",
    "# then the old": "loop 2: a flag, never a delete",
    "_retire(state, snap": "current=False, superseded_by, superseded_at, expire_at, effective_to",
    "pending == BATCH": "400 writes a commit: a long document flips in two",
    '"retired_at": firestore.SERVER_TIMESTAMP': "the undo window's clock, on the claim",
})

## 3. The source, annotated - `reactivate()`

Counted, dated, then flipped. The two checks came from a real failure: the first undo flipped whatever rows remained, and after the retention window that left a source with no current version.

In [ ]:
annotate(idempotency.reactivate, {
    "if withdrawn(": "the tombstone: make retire holds against the undo",
    "expected, stamp = ": "the claim's chunk count and its retire stamp",
    "age = (datetime.now": "days since the retirement",
    "short = expected": "rule 1: fewer rows than the claim counted",
    "if short or (age": "rule 2: retired longer ago than RETENTION_DAYS",
    '"event":"reactivate_incomplete"': "both numbers on the line; None comes back; nothing changed",
    '"reactivated_at": firestore.SERVER_TIMESTAMP': "the flip: retirement fields deleted, the clock the reader sorts by",
    '"status": "indexed"': "the claim back to indexed, retired_at cleared",
})

## 4. The claim race, simulated

`claim()`'s docstring says read-then-write without a transaction is "a race with a window measured in milliseconds - and Pub/Sub delivers duplicates concurrently, so it is a window that gets hit". Two threads on a plain dict, with a tiny sleep between the read and the write standing in for the network round trip, hit it almost every time. A lock - the transaction's role here - makes the read and the write one unit.

In [ ]:
import random, threading, time

def race(lock=None, trials=20):
    twice = 0
    for _ in range(trials):
        documents, wins = {}, []
        def worker():
            if lock: lock.acquire()
            try:
                if "acme_abc" not in documents:                 # the read
                    time.sleep(random.uniform(0, 0.002))        # the window: a network round trip
                    documents["acme_abc"] = "processing"        # the write
                    wins.append(1)
            finally:
                if lock: lock.release()
        threads = [threading.Thread(target=worker) for _ in range(2)]
        for t in threads: t.start()
        for t in threads: t.join()
        twice += len(wins) == 2
    return twice

both_won, one_won = race(lock=None), race(lock=threading.Lock())
print("without a transaction:", both_won, "of 20 trials indexed the document twice")
print("with one:            ", one_won, "of 20 trials indexed it twice")

## 5. `reactivate()`'s verdict, predicted

The two rules, copied from `idempotency.reactivate` (the `short = ...` line and the `if short or (age ...` line, lines 308-309 of the file), applied to four cases: a full undo inside the window; one row short; the window closed; and a claim from before the stamp, which has no clock, so the count alone decides. The cell also checks the copy against the source, so a change in the kit shows up here.

In [ ]:
RETENTION_DAYS = getattr(idempotency, "RETENTION_DAYS", 30)   # the same env read the worker does: 30 unless RETENTION_DAYS is set

def verdict(rows, chunks, age_days, window=RETENTION_DAYS):
    # idempotency.reactivate, lines 308-309
    short = chunks is not None and rows < int(chunks)
    if short or (age_days is not None and age_days > window):
        return "reactivate_incomplete: " + ("fewer rows than the claim counted" if short else "retired longer ago than the undo window")
    return f"reactivated: {rows} rows flipped, nothing embedded"

src = inspect.getsource(idempotency.reactivate)
if not ("len(rows) < int(expected)" in src and "age > window" in src):
    print("WARNING: the kit's reactivate() does not hold these two rules - a stale kit copy, or the rules moved: re-copy them")

for rows, chunks, age in [(3, 3, 2), (2, 3, 2), (3, 3, 31), (3, None, None)]:
    print(f"rows={rows} chunks={chunks!s:4} age_days={age!s:4} -> {verdict(rows, chunks, age)}")

## 6. On a deployed lane - the ledger of `acme`

**Needs a deployed lane.** Set `PROJECT_ID` above and run the auth cell. These cells read `sources/`, one `documents/` claim, `ledger/acme`, then recompute the fingerprint with the kit's own function and count current versus retired rows in `chunks/` per source. A few Firestore reads; nothing is written.

In [ ]:
# needs a deployed lane
from google.cloud import firestore
from google.cloud.firestore_v1.base_query import FieldFilter

TENANT = "acme"
db = firestore.Client(project=PROJECT_ID)
sources = sorted((s.to_dict() for s in db.collection("sources").where(filter=FieldFilter("tenant_id", "==", TENANT)).stream()),
                 key=lambda r: r.get("name") or "")
print(f"{'name':38} {'doc_key':12} {'generation':>16} {'chunks':>6} {'reused':>6} {'embedded':>8} {'retired':>7} status")
for r in sources:
    print(f"{(r.get('name') or '')[:38]:38} {(r.get('doc_key') or '')[:12]:12} {str(r.get('generation') or '')[-16:]:>16} "
          f"{r.get('chunks') or 0:>6} {str(r.get('reused', '-')):>6} {str(r.get('embedded', '-')):>8} {str(r.get('retired', '-')):>7} {r.get('status')}")
print(len(sources), "ledger rows")

In [ ]:
# needs a deployed lane: one claim, the per-version record behind the first ledger row
key = sources[0]["doc_key"]
claim = db.collection("documents").document(key).get().to_dict() or {}
print(key)
for k in ("status", "tenant_id", "chunks", "reused", "embedded", "generation", "claimed_at", "indexed_at", "retired_at", "lane"):
    if k in claim:
        print(f"  {k:12} {claim[k]}")

In [ ]:
# needs a deployed lane: ledger/acme against the kit's own corpus_fingerprint()
from idempotency import corpus_fingerprint

stored = db.collection("ledger").document(TENANT).get().to_dict() or {}
fp, n = corpus_fingerprint(db, TENANT)
print("stored:     ", stored.get("fingerprint"), "versions", stored.get("versions"), "last_event", stored.get("last_event"))
print("recomputed: ", fp, "versions", n)
print("match:", fp == stored.get("fingerprint"), "- a mismatch means something changed what is current after the last refresh")

In [ ]:
# needs a deployed lane: current versus retired rows per source (select() keeps the read light)
from collections import Counter

tally = Counter()
for snap in db.collection("chunks").where(filter=FieldFilter("tenant_id", "==", TENANT)).select(["source_uri", "current"]).stream():
    d = snap.to_dict() or {}
    tally[(d.get("source_uri", "?").rsplit("/", 1)[-1], "current" if d.get("current") else "retired or staged")] += 1

names = sorted({name for name, _ in tally})
print(f"{'source':40} {'current':>8} {'retired':>8}")
for name in names:
    print(f"{name[:40]:40} {tally[(name, 'current')]:>8} {tally[(name, 'retired or staged')]:>8}")
print("retired rows leave when their expire_at passes: the TTL policy in firestore_indexes.tf, RETENTION_DAYS after the swap")